# Intrusion Detection System - Model Training
## Using CIC-IDS 2017 Dataset with XGBoost

This notebook trains a binary classification model to detect network intrusions.
- **0 = BENIGN**
- **1 = ATTACK**

## 1. Import Libraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score
import xgboost as xgb
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print(f"XGBoost version: {xgb.__version__}")

Libraries imported successfully
XGBoost version: 3.1.3


## 2. Load Dataset

In [21]:
csv_files = [
    "../data/Monday-WorkingHours.pcap_ISCX.csv",
    "../data/Tuesday-WorkingHours.pcap_ISCX.csv",
    "../data/Wednesday-workingHours.pcap_ISCX.csv",
    "../data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "../data/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "../data/Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "../data/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "../data/Friday-WorkingHours-Afternoon-DDoS.pcap_ISCX.csv"
]

dfs = []
for file in csv_files:
    print(f"Loading {file}")
    dfs.append(pd.read_csv(file, low_memory=False))

df = pd.concat(dfs, ignore_index=True)
print("Final dataset shape:", df.shape)

Loading ../data/Monday-WorkingHours.pcap_ISCX.csv
Loading ../data/Tuesday-WorkingHours.pcap_ISCX.csv
Loading ../data/Wednesday-workingHours.pcap_ISCX.csv
Loading ../data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading ../data/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading ../data/Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading ../data/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading ../data/Friday-WorkingHours-Afternoon-DDoS.pcap_ISCX.csv
Final dataset shape: (2830743, 79)


In [25]:
df.columns = df.columns.str.strip()

print(df.columns)

Index(['Destination Port', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Total Length of Fwd Packets',
       'Total Length of Bwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
       'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Min Packet Length', 'Max Packet Length', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',
       'SYN Flag Co

## 3. Preprocessing

In [32]:
X = df.drop(columns=['Label'])
y = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

# Keep only numeric features
X = X.select_dtypes(include=["number"])

# Handle inf and NaN properly
import numpy as np
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


## 4. Train XGBoost

In [35]:
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective='binary:logistic',
    eval_metric='logloss'
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

## 5. Evaluation

In [38]:
y_pred = model.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9989508062365208
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    454620
           1       1.00      1.00      1.00    111529

    accuracy                           1.00    566149
   macro avg       1.00      1.00      1.00    566149
weighted avg       1.00      1.00      1.00    566149



## 6. Save Model

In [41]:
joblib.dump(model, '../models/ids_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
print('Model saved successfully')

Model saved successfully


In [43]:
import joblib

feature_names = X.columns.tolist()

joblib.dump(feature_names, "../models/feature_names.pkl")

print("feature_names.pkl saved successfully")
print("Number of features:", len(feature_names))



feature_names.pkl saved successfully
Number of features: 78
